In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import time
import random
from torchinfo import summary
import optuna
import tqdm as notebook_tqdm
from data_handler import *
from uncertainty import *
from linear_variational import *

# 4th GPU on AIMS02
device = torch.device('cuda:3')
print(f"Using device: {device}")
print(torch.cuda.get_device_name())

def set_random_seed(seed_value=42):
    # Python random seed
    random.seed(seed_value)
    
    # Numpy random seed
    np.random.seed(seed_value)
    
    # PyTorch seed
    torch.manual_seed(seed_value)
    
    # If using CUDA (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)  # if using multi-GPU
        torch.backends.cudnn.deterministic = True  # For reproducibility
        torch.backends.cudnn.benchmark = False  # Disable auto-optimization for determinism

set_random_seed()

Using device: cuda:3
NVIDIA RTX 6000 Ada Generation


In [9]:
# Load data
train = pd.read_csv('~/nureth/tf_and_ts/tuning/TF_TS_train.csv')
test = pd.read_csv('~/nureth/tf_and_ts/tuning/TF_TS_test.csv')
valid = pd.read_csv('~/nureth/tf_and_ts/tuning/TF_TS_valid.csv')

# Interpolate the NaN values
train['TS'] = train['TS'].interpolate()
test['TS'] = test['TS'].interpolate()
valid['TS'] = valid['TS'].interpolate()

# Function to find rows with NaN values in a pandas DataFrame
def check_nans_in_dataframe(df, column_name, name):
    nan_rows = df[df[column_name].isna()]
    if not nan_rows.empty:
        print(f"Rows with NaN values in {name} dataset:")
        print(nan_rows)
    else:
        print(f"No NaN values in {name} dataset.")

# Check for NaN values in each dataset
check_nans_in_dataframe(train, 'TS', 'Train')
check_nans_in_dataframe(test, 'TS', 'Test')
check_nans_in_dataframe(valid, 'TS', 'Validation')

No NaN values in Train dataset.
No NaN values in Test dataset.
No NaN values in Validation dataset.


In [10]:
# Preprocess the data
scaler = MinMaxScaler(feature_range=(0, 1))
train_data = scaler.fit_transform(train)
test_data = scaler.transform(test)
valid_data = scaler.transform(valid)

# Create sequences
def create_sequences(data, lookback=10):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i])  # Use the last 'lookback' time steps for prediction
        y.append(data[i])  # The next time step is the target
    return np.array(X), np.array(y)

timesteps = 10
Xtrain, Ytrain = create_sequences(train_data, lookback=timesteps)
Xtest, Ytest = create_sequences(test_data, lookback=timesteps)
Xvalid, Yvalid = create_sequences(valid_data, lookback=timesteps)

# Convert to PyTorch tensors
Xtrain_tensor = torch.tensor(Xtrain, dtype=torch.float32).to(device)
Ytrain_tensor = torch.tensor(Ytrain, dtype=torch.float32).to(device)
Xtest_tensor = torch.tensor(Xtest, dtype=torch.float32).to(device)
Ytest_tensor = torch.tensor(Ytest, dtype=torch.float32).to(device)
Xvalid_tensor = torch.tensor(Xvalid, dtype=torch.float32).to(device)
Yvalid_tensor = torch.tensor(Yvalid, dtype=torch.float32).to(device)

print(
    f"""Xtrain, Ytrain: {Xtrain_tensor.shape}, {Ytrain_tensor.shape}
Xtest,   Ytest: {Xtrain_tensor.shape}, {Ytrain_tensor.shape}
Xvalid, Yvalid: {Xvalid_tensor.shape}, {Yvalid_tensor.shape}"""
)

Xtrain, Ytrain: torch.Size([340060, 10, 2]), torch.Size([340060, 2])
Xtest,   Ytest: torch.Size([340060, 10, 2]), torch.Size([340060, 2])
Xvalid, Yvalid: torch.Size([170025, 10, 2]), torch.Size([170025, 2])


In [11]:
class vLSTM(nn.Module):
    def __init__(self, in_features, hidden_size1, hidden_size2, hidden_size3, out_features, prior_mean=0, prior_variance=0.5, posterior_rho_init=-4.0, bias=True):
        super(vLSTM, self).__init__()

        # Define multiple LSTM layers
        self.lstm1 = nn.LSTM(in_features, hidden_size1, batch_first=True)

        self.lstm2 = nn.LSTM(hidden_size1, hidden_size2, batch_first=True)

        self.lstm3 = nn.LSTM(hidden_size2, hidden_size3, batch_first=True)
        
        self.fc = LinearReparameterization(
            in_features=hidden_size3,
            out_features=out_features,
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_rho_init=posterior_rho_init,
            bias=bias
        )

    def forward(self, x, hidden_states=None):
        # LSTM1
        out, _ = self.lstm1(x, hidden_states)
    
        # LSTM2
        out, _ = self.lstm2(out, hidden_states)

        # LSTM3
        out, _ = self.lstm3(out, hidden_states)

        # Get the output for the **last time step** only: [batch_size, hidden_size]
        hidden_last_step = out[:, -1, :]  # Last time step

        # Pass through the final linear layer
        output, kl_fc = self.fc(hidden_last_step)
        kl_total = kl_fc

        # Return output and total KL divergence
        return output, kl_total

In [17]:
# Define the Optuna objective function using your training loop
def objective(trial):
    # --- Hyperparameter suggestions ---
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-1)
    batch_size = trial.suggest_categorical("batch_size", [256, 512])
    hidden_size1 = trial.suggest_int("hidden_size1", 16, 128, step=16)
    hidden_size2 = trial.suggest_int("hidden_size2", 16, 128, step=16)
    hidden_size3 = trial.suggest_int("hidden_size3", 8, 64, step=8)
    
    # Optionally, choose a KL schedule
    kl_schedule = "sigmoid_decay"

    # --- Instantiate the variational model ---
    # (Adjust in_features and out_features as needed for your data)
    in_features = 2
    out_features = 2
    model = vLSTM(
        in_features=in_features,
        hidden_size1=hidden_size1,
        hidden_size2=hidden_size2,
        hidden_size3=hidden_size3,
        out_features=out_features,
        prior_mean=0,
        prior_variance=0.5,
        posterior_rho_init=-4.0,
        bias=True
    ).to(device)

    # --- Optimizer and loss ---
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    reconstruction_loss_fn = nn.L1Loss()  # or nn.MSELoss()

    # --- DataLoaders for training and validation ---
    train_dataset = torch.utils.data.TensorDataset(Xtrain_tensor, Ytrain_tensor)
    val_dataset   = torch.utils.data.TensorDataset(Xvalid_tensor, Yvalid_tensor)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader   = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    num_epochs = 30
    # Track the validation loss from each epoch so that we can report to Optuna
    for epoch in range(num_epochs):
        model.train()

        # --- Update KL weight based on schedule ---
        if kl_schedule == 'linear':
            kl_weight = epoch / num_epochs
        elif kl_schedule == 'sigmoid_growth':
            kl_weight = 0.05 / (1 + np.exp(-2 * (epoch - 0.7 * num_epochs))) + 0.0005
        elif kl_schedule == 'sigmoid_decay':
            kl_weight = 0.05 / (1 + np.exp(2 * (epoch - 0.15 * num_epochs))) + 0.0005
        else:  # 'constant'
            kl_weight = 1.0

        running_train_loss = 0.0
        running_train_reconstruction_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs, kl_loss = model(inputs)
            reconstruction_loss = reconstruction_loss_fn(outputs, targets)
            total_loss = reconstruction_loss + kl_weight * kl_loss
            total_loss.backward()
            optimizer.step()
            running_train_reconstruction_loss += reconstruction_loss.item()
            running_train_loss += total_loss.item()
        avg_train_loss = running_train_loss / len(train_loader)
        avg_train_reconstruction_loss = running_train_reconstruction_loss / len(train_loader)

        # --- Validation Loop ---
        model.eval()
        running_val_loss = 0.0
        running_val_reconstruction_loss = 0.0
        with torch.no_grad():
            for val_inputs, val_targets in val_loader:
                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
                val_outputs, val_kl_loss = model(val_inputs)
                val_reconstruction_loss = reconstruction_loss_fn(val_outputs, val_targets)
                val_total_loss = val_reconstruction_loss + kl_weight * val_kl_loss
                running_val_reconstruction_loss += val_reconstruction_loss.item()
                running_val_loss += val_total_loss.item()
        avg_val_loss = running_val_loss / len(val_loader)
        avg_val_reconstruction_loss = running_val_reconstruction_loss / len(val_loader)

        print(f"Trial {trial.number}, Epoch {epoch+1}/{num_epochs}, "
              f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, KL Weight: {kl_weight:.4f}, Train Recon. Loss: {avg_train_reconstruction_loss:.4f}, Val Recon. Loss: {avg_val_reconstruction_loss} ")

        # Report intermediate validation loss for pruning
        trial.report(avg_val_reconstruction_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # Return the final validation loss (the lower the better)
    return avg_val_reconstruction_loss

In [18]:
# --- Run the study ---
trials = 20
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=trials)  # Use n_jobs>1 with caution on GPU

# Print out the best results
print("Best trial:")
best_trial = study.best_trial
print("  Final Validation Loss (MAE): {:.4f}".format(best_trial.value))
print("  Params: ")
for key, value in best_trial.params.items():
    print("    {}: {}".format(key, value))
print()

trials_df = study.trials_dataframe()
print(trials_df)

[I 2025-02-16 21:02:50,371] A new study created in memory with name: no-name-3f19f279-0431-4572-8be3-4263e342ad2f
/tmp/ipykernel_3407211/2338334924.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-1)


Trial 0, Epoch 1/30, Train Loss: 0.4398, Val Loss: 0.3896, KL Weight: 0.0505, Train Recon. Loss: 0.1502, Val Recon. Loss: 0.10112757560469814 
Trial 0, Epoch 2/30, Train Loss: 0.3900, Val Loss: 0.3834, KL Weight: 0.0505, Train Recon. Loss: 0.1029, Val Recon. Loss: 0.09725263719964358 
Trial 0, Epoch 3/30, Train Loss: 0.3787, Val Loss: 0.3702, KL Weight: 0.0502, Train Recon. Loss: 0.0952, Val Recon. Loss: 0.08778599121358928 
Trial 0, Epoch 4/30, Train Loss: 0.3473, Val Loss: 0.3326, KL Weight: 0.0481, Train Recon. Loss: 0.0773, Val Recon. Loss: 0.06343494339457324 
Trial 0, Epoch 5/30, Train Loss: 0.2681, Val Loss: 0.2665, KL Weight: 0.0371, Train Recon. Loss: 0.0614, Val Recon. Loss: 0.06030451413383847 
Trial 0, Epoch 6/30, Train Loss: 0.1367, Val Loss: 0.1355, KL Weight: 0.0139, Train Recon. Loss: 0.0591, Val Recon. Loss: 0.058067592152453086 
Trial 0, Epoch 7/30, Train Loss: 0.0759, Val Loss: 0.0764, KL Weight: 0.0029, Train Recon. Loss: 0.0599, Val Recon. Loss: 0.06048199775357459

[I 2025-02-16 21:05:30,451] Trial 0 finished with value: 0.021199531543806934 and parameters: {'lr': 3.838611641997633e-05, 'batch_size': 512, 'hidden_size1': 80, 'hidden_size2': 16, 'hidden_size3': 24}. Best is trial 0 with value: 0.021199531543806934.


Trial 0, Epoch 30/30, Train Loss: 0.0228, Val Loss: 0.0241, KL Weight: 0.0005, Train Recon. Loss: 0.0199, Val Recon. Loss: 0.021199531543806934 
Trial 1, Epoch 1/30, Train Loss: 0.2757, Val Loss: 0.2150, KL Weight: 0.0505, Train Recon. Loss: 0.1275, Val Recon. Loss: 0.1296699513323247 
Trial 1, Epoch 2/30, Train Loss: 0.2105, Val Loss: 0.2010, KL Weight: 0.0505, Train Recon. Loss: 0.1400, Val Recon. Loss: 0.13694383123296652 
Trial 1, Epoch 3/30, Train Loss: 0.2008, Val Loss: 0.1947, KL Weight: 0.0502, Train Recon. Loss: 0.1393, Val Recon. Loss: 0.13511208896790897 
Trial 1, Epoch 4/30, Train Loss: 0.2001, Val Loss: 0.1937, KL Weight: 0.0481, Train Recon. Loss: 0.1405, Val Recon. Loss: 0.13148993617846622 
Trial 1, Epoch 5/30, Train Loss: 0.1820, Val Loss: 0.1858, KL Weight: 0.0371, Train Recon. Loss: 0.1322, Val Recon. Loss: 0.13538980298303627 
Trial 1, Epoch 6/30, Train Loss: 0.1482, Val Loss: 0.1480, KL Weight: 0.0139, Train Recon. Loss: 0.1252, Val Recon. Loss: 0.12088999596729248

[I 2025-02-16 21:08:12,699] Trial 1 finished with value: 0.12264742959827722 and parameters: {'lr': 0.008297299238221282, 'batch_size': 512, 'hidden_size1': 64, 'hidden_size2': 112, 'hidden_size3': 64}. Best is trial 0 with value: 0.021199531543806934.


...

Trial 17, Epoch 1/30, Train Loss: 0.3432, Val Loss: 0.3091, KL Weight: 0.0505, Train Recon. Loss: 0.1024, Val Recon. Loss: 0.11306545051528995 


[I 2025-02-16 21:44:40,038] Trial 18 pruned. 


Trial 18, Epoch 1/30, Train Loss: 0.4061, Val Loss: 0.3870, KL Weight: 0.0505, Train Recon. Loss: 0.1210, Val Recon. Loss: 0.10355339342836795 
Trial 19, Epoch 1/30, Train Loss: 0.3712, Val Loss: 0.3504, KL Weight: 0.0505, Train Recon. Loss: 0.0966, Val Recon. Loss: 0.08202358282038144 
Trial 19, Epoch 2/30, Train Loss: 0.3128, Val Loss: 0.3054, KL Weight: 0.0505, Train Recon. Loss: 0.0506, Val Recon. Loss: 0.048698676871380915 
Trial 19, Epoch 3/30, Train Loss: 0.2807, Val Loss: 0.2851, KL Weight: 0.0502, Train Recon. Loss: 0.0307, Val Recon. Loss: 0.040251131201709124 
Trial 19, Epoch 4/30, Train Loss: 0.2613, Val Loss: 0.2710, KL Weight: 0.0481, Train Recon. Loss: 0.0311, Val Recon. Loss: 0.045496648461803006 
Trial 19, Epoch 5/30, Train Loss: 0.2045, Val Loss: 0.2019, KL Weight: 0.0371, Train Recon. Loss: 0.0333, Val Recon. Loss: 0.03300328256537844 
Trial 19, Epoch 6/30, Train Loss: 0.0957, Val Loss: 0.0995, KL Weight: 0.0139, Train Recon. Loss: 0.0319, Val Recon. Loss: 0.03560302

[I 2025-02-16 21:48:37,196] Trial 19 finished with value: 0.013183182211246173 and parameters: {'lr': 0.00012072285661277648, 'batch_size': 256, 'hidden_size1': 64, 'hidden_size2': 64, 'hidden_size3': 56}. Best is trial 9 with value: 0.011827676282246785.


Trial 19, Epoch 30/30, Train Loss: 0.0129, Val Loss: 0.0164, KL Weight: 0.0005, Train Recon. Loss: 0.0096, Val Recon. Loss: 0.013183182211246173 
Best trial:
  Final Validation Loss (MAE): 0.0118
  Params: 
    lr: 0.0001804978313277879
    batch_size: 256
    hidden_size1: 48
    hidden_size2: 64
    hidden_size3: 32

    number     value             datetime_start          datetime_complete  \
0        0  0.021200 2025-02-16 21:02:50.372783 2025-02-16 21:05:30.450920   
1        1  0.122647 2025-02-16 21:05:30.451800 2025-02-16 21:08:12.698868   
2        2  0.112908 2025-02-16 21:08:12.699495 2025-02-16 21:12:16.578546   
3        3  0.064287 2025-02-16 21:12:16.579867 2025-02-16 21:14:53.832103   
4        4  0.116382 2025-02-16 21:14:53.832747 2025-02-16 21:18:49.156545   
5        5  0.017414 2025-02-16 21:18:49.157497 2025-02-16 21:21:25.609852   
6        6  0.018771 2025-02-16 21:21:25.610698 2025-02-16 21:25:21.128646   
7        7  0.197662 2025-02-16 21:25:21.129857 2025-02

In [19]:
trials_df

,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_hidden_size1,params_hidden_size2,params_hidden_size3,params_lr,state
0,0,0.021200,2025-02-16 21:02:50.372783,2025-02-16 21:05:30.450920,0 days 00:02:40.078137,512,80,16,24,0.000038,COMPLETE
1,1,0.122647,2025-02-16 21:05:30.451800,2025-02-16 21:08:12.698868,0 days 00:02:42.247068,512,64,112,64,0.008297,COMPLETE
2,2,0.112908,2025-02-16 21:08:12.699495,2025-02-16 21:12:16.578546,0 days 00:04:03.879051,256,64,112,64,0.003008,COMPLETE
3,3,0.064287,2025-02-16 21:12:16.579867,2025-02-16 21:14:53.832103,0 days 00:02:37.252236,512,32,64,40,0.000011,COMPLETE
4,4,0.116382,2025-02-16 21:14:53.832747,2025-02-16 21:18:49.156545,0 days 00:03:55.323798,256,128,112,48,0.003726,COMPLETE
5,5,0.017414,2025-02-16 21:18:49.157497,2025-02-16 21:21:25.609852,0 days 00:02:36.452355,512,48,80,24,0.000447,COMPLETE
6,6,0.018771,2025-02-16 21:21:25.610698,2025-02-16 21:25:21.128646,0 days 00:03:55.517948,256,96,128,16,0.000016,COMPLETE
7,7,0.197662,2025-02-16 21:25:21.129857,2025-02-16 21:25:28.984089,0 days 00:00:07.854232,256,96,112,24,0.018946,PRUNED
8,8,0.025508,2025-02-16 21:25:28.985034,2025-02-16 21:28:06.432743,0 days 00:02:37.447709,512,96,96,64,0.000147,COMPLETE
9,9,0.011828,2025-02-16 21:28:06.433645,2025-02-16 21:32:00.735398,0 days 00:03:54.301753,256,48,64,32,0.000180,COMPLETE


In [20]:
trials_df_sorted = trials_df.sort_values(by='value')
trials_df_sorted

,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_hidden_size1,params_hidden_size2,params_hidden_size3,params_lr,state
9,9,0.011828,2025-02-16 21:28:06.433645,2025-02-16 21:32:00.735398,0 days 00:03:54.301753,256,48,64,32,0.000180,COMPLETE
14,14,0.012113,2025-02-16 21:40:01.564291,2025-02-16 21:43:52.883153,0 days 00:03:51.318862,256,48,64,40,0.000076,COMPLETE
19,19,0.013183,2025-02-16 21:44:40.038799,2025-02-16 21:48:37.196604,0 days 00:03:57.157805,256,64,64,56,0.000121,COMPLETE
13,13,0.013954,2025-02-16 21:36:11.661122,2025-02-16 21:40:01.563136,0 days 00:03:49.902014,256,32,64,40,0.000077,COMPLETE
10,10,0.017407,2025-02-16 21:32:00.736561,2025-02-16 21:35:55.971122,0 days 00:03:55.234561,256,16,48,8,0.000368,COMPLETE
5,5,0.017414,2025-02-16 21:18:49.157497,2025-02-16 21:21:25.609852,0 days 00:02:36.452355,512,48,80,24,0.000447,COMPLETE
6,6,0.018771,2025-02-16 21:21:25.610698,2025-02-16 21:25:21.128646,0 days 00:03:55.517948,256,96,128,16,0.000016,COMPLETE
0,0,0.021200,2025-02-16 21:02:50.372783,2025-02-16 21:05:30.450920,0 days 00:02:40.078137,512,80,16,24,0.000038,COMPLETE
8,8,0.025508,2025-02-16 21:25:28.985034,2025-02-16 21:28:06.432743,0 days 00:02:37.447709,512,96,96,64,0.000147,COMPLETE
3,3,0.064287,2025-02-16 21:12:16.579867,2025-02-16 21:14:53.832103,0 days 00:02:37.252236,512,32,64,40,0.000011,COMPLETE
